# FASTT — main results on Earnings-22

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/fastt/blob/main/notebooks/main_earnings22.ipynb)

This notebook reproduces the paper's main experiment on Earnings-22. Every method's hyperparameters are selected on
a 25 % hold-out of the corpus; every method is then trained and tested on the same ten folds of a 5×2
cross-validation of the remaining clips, and the two FASTT variants are compared with every other method on those
folds.

The configuration is fixed in `fastt.configs.MAIN["earnings22"]` and the code is in `fastt.experiment`. The
features and the stored searches and folds are read from the Hugging Face dataset
[`huseyin-karaca/fastt`](https://huggingface.co/datasets/huseyin-karaca/fastt); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold refitted from the published features (`RETUNE = True`: the search as well) | a CPU, hours |
| **3** | level 2 on features rebuilt from the audio: run `extract` first on the same machine, which writes them to a local folder that level 3 reads (`FASTT_HUB` names another place) | a GPU for `extract` |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastt") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastt[paper] @ git+https://github.com/huseyin-karaca/fastt.git@main"], check=True)

In [2]:
from fastt.configs import MAIN
from fastt.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again

exp = MainExperiment(MAIN["earnings22"], level=LEVEL, retune=RETUNE)

Earnings-22: level 1 — search read from the Hub, folds read from the Hub
experts  data2vec-audio-base, ESPnet Conformer-RNN-T, wav2vec2-large-960h
arms     FASTT-Alternating, FASTT-Boosted, Raw XGBoost (AdaSTT), Raw SDT, SelectKBest XGBoost, SelectKBest SDT, ROVER (weighted), ROVER (confidence-weighted), CN-MBR, Score-Fusion, Hard-Selector (plain CE), Hard-Selector (cost-sensitive CE), Soft-Selector (mixture), QE-Router


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every method and their choices; every other parameter is fixed
(`fastt.configs.SPACES`). The spaces are the same on every corpus.

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed
  [FASTT-Alternating  (18 free of 18)]
           parameter kind                                   choices / range
      transform_type  cat {nonlinear, linear, diagonal, low_rank, identity}
      num_iterations  cat                                     {6, 12, 3, 1}
        transform_lr  cat                             {0.001, 0.0001, 0.01}
     transform_steps  cat                                    {100, 25, 400}
        surrogate_lr  cat                                {0.01, 0.1, 0.001}
     surrogate_steps  cat                                    {50, 200, 400}
             lambda1  cat                               {0.1, 0.001, 1e-05}
             lambda2  cat                               {1e-05, 0.1, 0.001}
    sel_n_estimators  cat                                             {400}
   sel_learning_rate  cat                            {0.03, 0.3, 0.1, 0.01}
       sel_max_depth  cat                                     {5, 12, 8, 3}
       sel_subsample  cat                                   {1.0, 0.6, 0.8}
       sel_colsample  cat                              {0.6, 1.0, 0.3, 0.1}
sel_min_child_weight  cat                                             {1.0}
      sel_reg_lambda  cat                                             {3.0}
        weight_decay  cat                                            {0.01}
           out_ratio  cat                                   {1.0, 0.1, 0.3}
    bottleneck_ratio  cat                                            {0.05}
  [FASTT-Boosted  (15 free of 15)]
        parameter kind                                   choices / range
   transform_type  cat {identity, low_rank, linear, nonlinear, diagonal}
          lambda1  cat                                           {0.001}
          lambda2  cat                               {0.001, 1e-05, 0.1}
        out_ratio  cat                                             {0.1}
 bottleneck_ratio  cat                                 {0.15, 0.5, 0.05}
       num_rounds  cat                                      {6, 1, 4, 2}
        num_trees  cat                                    {24, 8, 16, 4}
            depth  cat                                               {7}
            lmbda  cat                        {0.01, 0.0001, 1.0, 1e-06}
           lmbda2  cat                                           {1e-07}
          adam_lr  cat                                          {0.0003}
adam_weight_decay  cat                        {0.001, 1e-07, 1e-05, 0.1}
       batch_size  cat                                  {512, 128, 2048}
         patience  cat                                      {40, 20, 10}
   t_weight_decay  cat                                           {1e-06}
  [Raw XGBoost (AdaSTT)  (7 free of 7)]
           parameter kind        choices / range
    sel_n_estimators  cat   {1000, 400, 150, 50}
   sel_learning_rate  cat {0.3, 0.1, 0.03, 0.01}
       sel_max_depth  cat                    {3}
       sel_subsample  cat        {1.0, 0.8, 0.6}
       sel_colsample  cat                  {1.0}
sel_min_child_weight  cat                 {16.0}
      sel_reg_lambda  cat       {3.0, 0.3, 30.0}
  [Raw SDT  (14 free of 15)]
        parameter kind              choices / range
   t_weight_decay  cat                      {1e-06}
          lambda1  cat                      {0.001}
          lambda2  cat                      {0.001}
        out_ratio  cat                        {0.1}
 bottleneck_ratio  cat                       {0.05}
       num_rounds  cat                 {1, 2, 4, 6}
        num_trees  cat                          {8}
            depth  cat                 {2, 7, 5, 3}
            lmbda  cat   {1e-06, 1.0, 0.0001, 0.01}
           lmbda2  cat                        {0.1}
          adam_lr  cat {0.001, 0.003, 0.0003, 0.01}
adam_weight_decay  cat                      {1e-07}
       batch_size  cat             {2048, 512, 128}
         patience  cat              

## 2. Data

The three experts' pooled encoder features, their transcripts, and the transcripts' word-error counters against
the reference, for every clip. A sample of the stored counters is recomputed from the stored transcripts as a
check.

In [4]:
exp.load()

12,088 utterances x 3 routable model(s), from cache .fastt-cache/store_1ca812b368b89821
texts for 3 model(s), from cache
100.0% of 1,200 (utterance, model) counters reproduce exactly


EnsembleStore(earnings22, 'mean', 12,088 utterances, 3 routable)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every method, ten configurations are drawn at random
from its space (Table 2), each fitted on 70 % of the hold-out and scored on the remaining 30 %; the configuration
with the lowest corpus WER is kept. When several tie on it, the one the paper's run kept is used
(`fastt.configs.SELECTED`), after checking that it is among the tied best. The other 75 % of the clips are never
seen by the search.

In [5]:
exp.search()


=== best fastt_alternating   corpus WER 0.4036   gap closed +34.4%   (10 trials, 0.0 min)
{
  "num_iterations": 1,
  "sel_colsample": 0.6,
  "sel_learning_rate": 0.03,
  "sel_max_depth": 5,
  "sel_min_child_weight": 1.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 3.0,
  "sel_subsample": 0.8,
  "transform_type": "identity"
}

=== best fastt_boosted   corpus WER 0.4037   gap closed +34.2%   (10 trials, 0.0 min)
{
  "adam_lr": 0.0003,
  "adam_weight_decay": 1e-07,
  "batch_size": 2048,
  "depth": 7,
  "epochs": 200,
  "lambda1": 0.001,
  "lambda2": 0.001,
  "lmbda": 0.0001,
  "lmbda2": 1e-07,
  "num_rounds": 6,
  "num_trees": 24,
  "patience": 40,
  "transform_type": "diagonal"
}

=== best adastt   corpus WER 0.4116   gap closed +22.8%   (10 trials, 0.0 min)
{
  "sel_colsample": 1.0,
  "sel_learning_rate": 0.1,
  "sel_max_depth": 3,
  "sel_min_child_weight": 16.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 30.0,
  "sel_subsample": 1.0
}

=== best raw_sdt   corpus WER 0.4217   gap


=== best rover_w   corpus WER 0.4322   gap closed -6.9%   (10 trials, 0.0 min)
{
  "alpha": 1.0,
  "conf_source": "cn",
  "null_penalty": 0.5,
  "weight_by_conf": false,
  "weighting": "log_odds"
}

=== best rover_conf   corpus WER 0.4362   gap closed -12.6%   (10 trials, 0.0 min)
{
  "alpha": 0.25,
  "conf_source": "cn",
  "conf_temp": 0.25,
  "null_penalty": 0.25,
  "weight_by_conf": false,
  "weighting": "log_odds"
}

=== best cn_mbr   corpus WER 0.4421   gap closed -21.1%   (10 trials, 0.0 min)
{
  "candidates": "members",
  "conf_source": "probe",
  "decode": "hyp",
  "null_prior": 0.25,
  "prior": "inverse_wer",
  "temperature": 10.0,
  "use_conf": true
}

=== best score_fusion   corpus WER 0.4421   gap closed -21.2%   (10 trials, 0.0 min)
{
  "mode": "hard",
  "w_conf": 1.0,
  "w_consensus": -1.0,
  "w_length": -1.0
}

=== best argmin_clf   corpus WER 0.4124   gap closed +21.7%   (10 trials, 0.0 min)
{
  "alpha": 1.0
}

=== best probe   corpus WER 0.4201   gap closed +10.7%   (

partition,n_utts,share
tune,3022,0.250
eval,9066,0.750
fit,2115,0.175
val,907,0.075


  on the tuning-score rows: bsm(wav2vec2_large) 0.4275   oracle 0.3581   gap 0.0693
  the 9,066 evaluation rows are never seen by a trial
  objectives   : corpus_wer (minimize) — crossval.METRICS['corpus_wer'], on the tuning-score rows
  14 arms x 10 trials, read from the Hub; 9,066 evaluation clips the search never saw


HPTRun('hpt_earnings22_8657033e', 6 arm(s), best fastt_alternating 0.4036, eval on 9,066 rows)

In [6]:
exp.dataset()

statistic,earnings22
Total clips,12088
Features,2305
Hyperparameter hold-out,3022
Evaluation rows (5x2),9066
Train size (per fold),4533
Test size (per fold),4533
Hours (est.),25.7386
Reference words,217118


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
FASTT-Alternating,10,0.4036,"{""num_iterations"": 1, ""sel_colsample"": 0.6, ""sel_learning_rate"": 0.03, ""sel_max_depth"": 5, ""sel_min_child_weight"": 1.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 3.0, ""sel_subsample"": 0.8, ""transform_type"": ""identity""}"
FASTT-Boosted,10,0.4037,"{""adam_lr"": 0.0003, ""adam_weight_decay"": 1e-07, ""batch_size"": 2048, ""depth"": 7, ""epochs"": 200, ""lambda1"": 0.001, ""lambda2"": 0.001, ""lmbda"": 0.0001, ""lmbda2"": 1e-07, ""num_rounds"": 6, ""num_trees"": 24, ""patience"": 40, ""transform_type"": ""diagonal""}"
Raw XGBoost (AdaSTT),10,0.4116,"{""sel_colsample"": 1.0, ""sel_learning_rate"": 0.1, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 30.0, ""sel_subsample"": 1.0}"
Raw SDT,10,0.4217,"{""adam_lr"": 0.001, ""adam_weight_decay"": 1e-07, ""batch_size"": 512, ""depth"": 5, ""epochs"": 200, ""lmbda"": 0.0001, ""lmbda2"": 0.1, ""num_rounds"": 2, ""num_trees"": 8, ""patience"": 20, ""transform_type"": ""identity""}"
SelectKBest XGBoost,10,0.4149,"{""k"": 512, ""sel_colsample"": 0.1, ""sel_learning_rate"": 0.01, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 30.0, ""sel_subsample"": 1.0}"
SelectKBest SDT,10,0.4217,"{""adam_lr"": 0.003, ""adam_weight_decay"": 0.1, ""batch_size"": 2048, ""depth"": 7, ""epochs"": 200, ""k"": 32, ""lmbda"": 1e-06, ""lmbda2"": 0.1, ""num_rounds"": 4, ""num_trees"": 24, ""patience"": 20, ""transform_type"": ""identity""}"
ROVER (weighted),10,0.4322,"{""alpha"": 1.0, ""conf_source"": ""cn"", ""null_penalty"": 0.5, ""weight_by_conf"": false, ""weighting"": ""log_odds""}"
ROVER (confidence-weighted),10,0.4362,"{""alpha"": 0.25, ""conf_source"": ""cn"", ""conf_temp"": 0.25, ""null_penalty"": 0.25, ""weight_by_conf"": false, ""weighting"": ""log_odds""}"
CN-MBR,10,0.4421,"{""candidates"": ""members"", ""conf_source"": ""probe"", ""decode"": ""hyp"", ""null_prior"": 0.25, ""prior"": ""inverse_wer"", ""temperature"": 10.0, ""use_conf"": true}"
Score-Fusion,10,0.4421,"{""mode"": ""hard"", ""w_conf"": 1.0, ""w_consensus"": -1.0, ""w_length"": -1.0}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every method is trained
on one half and tested on the other, then the other way round: ten folds, the same for every method.

In [8]:
exp.cross_validate();

  runlog HIT  cv5x2 seed 11 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 22 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 33 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 44 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 55 — 15/15 group(s) cached, nothing refitted


5 repetitions x 2 folds x 24 arms, read from the Hub


## 5. Main results (Table 3)

Corpus WER and mean utterance WER (mean ± sd over the ten folds), the share of the oracle gap closed and the
selection accuracy at 0, 1 and 2 words of tolerance, and how often each method picks each expert.

In [9]:
exp.results()

Table 3 — main results, Earnings-22
mean ± sd over 10 folds; GC / uGC / Acc in %; dist = share of clips routed to data2vec-audio-base / ESPnet Conformer-RNN-T / wav2vec2-large-960h; best non-oracle value per column in bold
  [individual]
                                       arm             WER            uWER        GC@0        GC@1        GC@2        uGC@0        uGC@1        uGC@2      Acc@0      Acc@1      Acc@2           dist
                       data2vec-audio-base 0.4506 ± 0.0042 0.5572 ± 0.0067 -61.9 ± 2.9 -74.3 ± 3.3 -98.8 ± 4.3 -37.6 ± 16.9 -51.5 ± 22.5 -78.6 ± 32.3 34.4 ± 0.5 56.2 ± 0.6 72.6 ± 0.6 1.00-0.00-0.00
                    ESPnet Conformer-RNN-T 0.4096 ± 0.0032 0.5092 ± 0.0058   1.1 ± 1.4   1.3 ± 1.7   1.7 ± 2.3  10.7 ± 11.3  14.9 ± 15.8  23.7 ± 25.1 54.7 ± 0.5 73.0 ± 0.3 84.5 ± 0.3 0.00-1.00-0.00
                       wav2vec2-large-960h 0.4097 ± 0.0044 0.5320 ± 0.0064   0.9 ± 1.3   1.0 ± 1.5   1.4 ± 2.0 -12.2 ± 12.9 -16.5 ± 17.6 -24.8 ± 26.4 52.4 ± 0.8 72.9 ± 0.4 85.1 ± 0.3 0.00-0.00-1.00
Best-Single-Model (ESPnet Conformer-RNN-T) 0.4103 ± 0.0039 0.5211 ± 0.0095   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0    0.0 ± 0.0    0.0 ± 0.0    0.0 ± 0.0 53.2 ± 1.4 72.6 ± 0.2 84.7 ± 0.3 0.00-0.50-0.50
  [external]
                              arm             WER            uWER        GC@0        GC@1        GC@2        uGC@0        uGC@1        uGC@2      Acc@0      Acc@1      Acc@2           dist
                  Random-Ensemble 0.4242 ± 0.0040 0.5338 ± 0.0072 -21.4 ± 2.4 -25.7 ± 2.7 -34.1 ± 3.5 -14.1 ± 14.4 -19.2 ± 19.5 -28.8 ± 29.3 46.8 ± 0.8 67.2 ± 0.4 80.6 ± 0.2 0.33-0.33-0.33
                 ROVER (weighted) 0.4151 ± 0.0062 0.5423 ± 0.0062  -7.3 ± 5.0  -8.8 ± 6.1 -11.8 ± 8.2 -22.1 ± 10.2 -30.3 ± 13.6 -46.2 ± 19.6          —          —          —              —
      ROVER (confidence-weighted) 0.4241 ± 0.0045 0.5507 ± 0.0067 -21.2 ± 1.6 -25.5 ± 2.0 -33.9 ± 2.8 -30.8 ± 14.1 -42.2 ± 18.8 -64.4 ± 27.1          —          —          —              —
                           CN-MBR 0.4295 ± 0.0048 0.5501 ± 0.0066 -29.5 ± 3.1 -35.4 ± 3.7 -47.1 ± 4.8 -30.3 ± 15.4 -41.5 ± 20.6 -63.3 ± 29.8 47.1 ± 0.6 67.1 ± 0.7 79.6 ± 0.6 0.19-0.38-0.43
                     Score-Fusion 0.4280 ± 0.0032 0.5378 ± 0.0045 -27.2 ± 2.7 -32.7 ± 3.3 -43.5 ± 4.5 -18.1 ± 14.5 -24.7 ± 19.7 -37.4 ± 29.2 44.9 ± 0.7 64.3 ± 0.5 78.7 ± 0.5 0.27-0.42-0.32
         Hard-Selector (plain CE) 0.3971 ± 0.0032 0.5003 ± 0.0068  20.3 ± 2.3  24.4 ± 3.0  32.4 ± 4.0  19.5 ± 11.3  27.0 ± 15.9  42.4 ± 25.8 57.2 ± 0.7 76.1 ± 0.6 88.2 ± 0.5 0.31-0.52-0.17
Hard-Selector (cost-sensitive CE) 0.4085 ± 0.0030 0.5084 ± 0.0060   2.8 ± 2.1   3.4 ± 2.6   4.5 ± 3.4  11.4 ± 11.3  16.0 ± 15.8  25.3 ± 25.1 55.2 ± 0.7 73.5 ± 0.6 85.0 ± 0.5 0.00-0.96-0.04
                        QE-Router 0.4056 ± 0.0035 0.5070 ± 0.0053   7.2 ± 2.5   8.7 ± 3.1  11.6 ± 4.1  12.9 ± 10.8  17.9 ± 15.2  28.3 ± 24.3 54.1 ± 0.7 73.3 ± 0.4 85.3 ± 0.5 0.25-0.37-0.37
  [classical]
                 arm             WER            uWER       GC@0       GC@1       GC@2       uGC@0       uGC@1       uGC@2      Acc@0      Acc@1      Acc@2           dist
Raw XGBoost (AdaSTT) 0.3926 ± 0.0034 0.4945 ± 0.0063 27.2 ± 1.5 32.6 ± 1.9 43.4 ± 2.6  25.5 ± 9.2 35.3 ± 13.2 55.1 ± 22.0 60.2 ± 0.6 78.7 ± 0.4 89.4 ± 0.3 0.02-0.48-0.50
             Raw SDT 0.4096 ± 0.0032 0.5092 ± 0.0058  1.1 ± 1.4  1.3 ± 1.7  1.7 ± 2.3 10.7 ± 11.3 14.9 ± 15.8 23.7 ± 25.1 54.7 ± 0.5 73.0 ± 0.3 84.5 ± 0.3 0.00-1.00-0.00
 SelectKBest XGBoost 0.3975 ± 0.0028 0.5011 ± 0.0072 19.6 ± 4.3 23.6 ± 5.3 31.4 ± 7.3 18.7 ± 11.9 26.0 ± 16.8 40.8 ± 27.1 58.8 ± 0.7 77.4 ± 0.9 88.4 ± 0.7 0.00-0.62-0.38
     SelectKBest SDT 0.4096 ± 0.0032 0.5092 ± 0.0058  1.1 ± 1.4  1.3 ± 1.7  1.7 ± 2.3 10.7 ± 11.3 14.9 ± 15.8 23.7 ± 25.1 54.7 ± 0.5 73.0 ± 0.3 84.5 ± 0.3 0.00-1.00-0.00
  [fastt]
              arm             WER            uWER       GC@0       GC@1       GC@2      uGC@0       uGC@1       uGC@2      Acc@0      Acc@1      Acc@2           dist
    FASTT-Boosted 

In [10]:
exp.routing()

model,n_picks,n_fused,share_data2vec,share_espnet_transducer,share_wav2vec2_large,top_member,share_top,entropy,entropy_max,share_sd_mean,switch_rate
achievable_oracle,45330,0,0.0487,0.4759,0.4755,espnet_transducer,0.4759,1.2320,1.5850,0.2627,0.1528
adastt,45330,0,0.0176,0.4778,0.5047,wav2vec2_large,0.5047,1.1094,1.5850,0.0312,0.4840
argmin_clf,45330,0,0.3095,0.5217,0.1688,espnet_transducer,0.5217,1.4467,1.5850,0.0170,0.6551
bsm,45330,0,0.0000,0.5000,0.5000,espnet_transducer,0.5000,1.0000,1.5850,0.3514,0.0000
cn_mbr,45330,0,0.1870,0.3816,0.4314,wav2vec2_large,0.4314,1.5060,1.5850,0.0063,0.5874
data2vec,45330,0,1.0000,0.0000,0.0000,data2vec,1.0000,0.0000,1.5850,0.0000,1.0000
espnet_transducer,45330,0,0.0000,1.0000,0.0000,espnet_transducer,1.0000,0.0000,1.5850,0.0000,0.5000
fastt_alternating,45330,0,0.0192,0.4325,0.5483,wav2vec2_large,0.5483,1.1079,1.5850,0.0231,0.4944
fastt_boosted,45330,0,0.0622,0.4824,0.4554,espnet_transducer,0.4824,1.2735,1.5850,0.0203,0.5169
kbest_sdt,45330,0,0.0000,1.0000,0.0000,espnet_transducer,1.0000,0.0000,1.5850,0.0000,0.5000


## 6. Significance (Table 6)

Every method against the FASTT anchor (the better of the two FASTT variants on corpus WER) on the same ten folds:
Dietterich's 5×2 cv paired $t$ test and Alpaydın's combined 5×2 cv $F$ test, on corpus WER and on mean utterance
WER.

In [11]:
exp.tests()

Table 6 — every arm against the FASTT anchor, Earnings-22
on the same 10 folds; positive effect = the anchor is better, in the metric's own units; p < 0.05 in bold. The F-test is the one the paper reports.
  [mean utterance WER — anchor FASTT-Alternating]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
              data2vec-audio-base +0.0671       11.628       0.0001    149.082     0.0000
           ESPnet Conformer-RNN-T +0.0190        8.130       0.0005     66.351     0.0001
              wav2vec2-large-960h +0.0419       11.996       0.0001    137.232     0.0000
                Best-Single-Model +0.0309        1.251       0.2664      4.615     0.0526
                  Random-Ensemble +0.0437       10.391       0.0001    114.646     0.0000
                 ROVER (weighted) +0.0522        8.276       0.0004     77.349     0.0001
      ROVER (confidence-weighted) +0.0606       14.276       0.0000    211.348     0.0000
                           CN-MBR +0.0599       14.546       0.0000    212.723     0.0000
                     Score-Fusion +0.0477        7.044       0.0009     60.920     0.0001
         Hard-Selector (plain CE) +0.0102        3.548       0.0164     10.361     0.0093
Hard-Selector (cost-sensitive CE) +0.0183        7.586       0.0006     55.710     0.0002
                        QE-Router +0.0169        3.985       0.0105     25.808     0.0011
             Raw XGBoost (AdaSTT) +0.0044        3.784       0.0128     10.848     0.0084
                          Raw SDT +0.0190        8.130       0.0005     66.351     0.0001
              SelectKBest XGBoost +0.0109        3.442       0.0184      9.276     0.0120
                  SelectKBest SDT +0.0190        8.130       0.0005     66.351     0.0001
                    FASTT-Boosted +0.0005       -0.056       0.9578      0.973     0.5482
            Oracle (no tolerance) -0.0698      -24.777       0.0000    576.722     0.0000
        Oracle (1 word tolerance) -0.0420       -7.703       0.0006     45.815     0.0003
        Oracle (2 word tolerance) -0.0162       -2.275       0.0720      2.594     0.1522
  [corpus WER — anchor FASTT-Alternating]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
              data2vec-audio-base +0.0613       19.133       0.0000    389.249     0.0000
           ESPnet Conformer-RNN-T +0.0203       13.754       0.0000    168.063     0.0000
              wav2vec2-large-960h +0.0204       11.061       0.0001    130.120     0.0000
                Best-Single-Model +0.0210       16.088       0.0000    246.158     0.0000
                  Random-Ensemble +0.0349       17.301       0.0000    298.603     0.0000
                 ROVER (weighted) +0.0258        4.753       0.0051     28.169     0.0009
      ROVER (confidence-weighted) +0.0348       14.542       0.0000    222.631     0.0000
                           CN-MBR +0.0402       10.953       0.0001    136.426     0.0000
                     Score-Fusion +0.0387       16.236       0.0000    281.293     0.0000
         Hard-Selector (plain CE) +0.0078        6.929       0.0010     36.180     0.0005
Hard-Selector (cost-sensitive CE) +0.0192        8.986       0.0003     68.661     0.0001
                        QE-Router +0.0163        5.429       0.0029     30.770     0.0007
             Raw XGBoost (AdaSTT) +0.0033        5.890       0.0020     27.793     0.0009
                          Raw SDT +0.0203       13.754       0.0000    168.063     0.0000
              SelectKBest XGBoost +0.0082        3.209       0.0237      6.377     0.0271
                  SelectKBest SDT +0.0203       13.754       0.0000    168.063     0.0000
                    FASTT-Boosted +0.0008        0.876       0.4213      1.406     0.3712
            Oracle (no tolerance) -0.0441      -40.869       0.0000   1687.257     0.0000
        Oracle (1 word tolerance) -0.0332      -32.524       0.0000   1033.096     0.0000
        Orac

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_earnings22.json`;
`manuscript_tables` reads it from the folder `FASTT_RECORDS_DIR` names.

In [12]:
exp.save();

wrote main_earnings22.json
